1: MNIST Data Preparation and Client Partitioning


In [ ]:
# ============================================================
# Purpose:
# - Create deterministic MNIST client splits for federated learning
# - Simulate heterogeneous clients by removing 3 to 5 classes per client
# - Build train/validation datasets for each client
# - Plot per-client label distributions for inspection
#
# Outputs:
#   client_train_datasets
#   client_val_datasets
#   test_ds
#   client_idxs
#   client_present
#   y_train
# ============================================================

import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# ---------------- Configuration ----------------
NUM_CLIENTS = 10
NUM_CLASSES = 10
BATCH_SIZE = 64
VAL_FRAC = 0.1
SEED = 42

MISSING_MIN = 3
MISSING_MAX = 5
ALPHA_WITHIN = 0.2

# Set random seeds for reproducibility
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- Client Partitioning ----------------
def make_clients_missing_classes(y, num_clients, num_classes,
                                 missing_min, missing_max,
                                 alpha_within, seed):
    """
    Create deterministic client splits with missing classes.

    Each client is assigned a subset of classes, with the number of
    missing classes sampled between missing_min and missing_max.
    For each class, samples are distributed only among clients that
    contain that class, using a Dirichlet allocation.

    Parameters
    ----------
    y : np.ndarray
        Training labels.
    num_clients : int
        Number of clients.
    num_classes : int
        Total number of classes.
    missing_min : int
        Minimum number of classes missing per client.
    missing_max : int
        Maximum number of classes missing per client.
    alpha_within : float
        Dirichlet concentration parameter controlling within-class skew.
    seed : int
        Random seed for deterministic splits.

    Returns
    -------
    client_idxs : list of lists
        Sample indices assigned to each client.
    client_present : list of lists
        Classes available on each client.
    """
    rng = np.random.default_rng(seed)
    y = y.reshape(-1)

    # Select the set of classes available on each client
    client_present = []
    for _ in range(num_clients):
        m = int(rng.integers(missing_min, missing_max + 1))
        present = rng.choice(num_classes, size=num_classes - m, replace=False)
        client_present.append(sorted(present.tolist()))

    # Build a lookup from class to eligible clients
    class_to_clients = {c: [] for c in range(num_classes)}
    for k, cls_list in enumerate(client_present):
        for c in cls_list:
            class_to_clients[c].append(k)

    # Distribute each class only to clients that contain it
    client_idxs = [[] for _ in range(num_clients)]
    for c in range(num_classes):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)

        eligible_clients = class_to_clients[c]
        if len(eligible_clients) == 0:
            raise RuntimeError(f"No clients contain class {c}. Please adjust the parameters or seed.")

        proportions = rng.dirichlet(alpha_within * np.ones(len(eligible_clients)))
        cuts = (np.cumsum(proportions) * len(idx)).astype(int)
        splits = np.split(idx, cuts[:-1])

        for j, k in enumerate(eligible_clients):
            client_idxs[k].extend(splits[j].tolist())

    # Shuffle each client's final index list
    for k in range(num_clients):
        rng.shuffle(client_idxs[k])

    return client_idxs, client_present


def make_tf_dataset(X, y, idx, batch_size=64, shuffle=True):
    """
    Build a TensorFlow dataset from a subset of samples.
    """
    ds = tf.data.Dataset.from_tensor_slices((X[idx], y[idx]))
    if shuffle:
        ds = ds.shuffle(min(len(idx), 20000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def build_mnist_clients():
    """
    Load MNIST, apply preprocessing, split the data across clients,
    and create train/validation/test datasets.
    """
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()
    y_tr = y_tr.reshape(-1).astype("int32")
    y_te = y_te.reshape(-1).astype("int32")

    # Scale pixel values to [0, 1] and add a channel dimension
    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    # Normalize using training-set statistics
    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7
    x_tr = (x_tr - mean) / std
    x_te = (x_te - mean) / std

    # Create deterministic heterogeneous client partitions
    client_idxs, client_present = make_clients_missing_classes(
        y_tr,
        num_clients=NUM_CLIENTS,
        num_classes=NUM_CLASSES,
        missing_min=MISSING_MIN,
        missing_max=MISSING_MAX,
        alpha_within=ALPHA_WITHIN,
        seed=SEED
    )

    client_train_datasets, client_val_datasets = [], []

    for k in range(NUM_CLIENTS):
        idx = np.array(client_idxs[k])
        n = len(idx)

        if n < 10:
            raise RuntimeError(f"Client {k} has too few samples ({n}). Try a different alpha or seed.")

        n_val = max(1, int(VAL_FRAC * n))
        val_idx = idx[:n_val]
        tr_idx = idx[n_val:]

        client_train_datasets.append(
            make_tf_dataset(x_tr, y_tr, tr_idx, batch_size=BATCH_SIZE, shuffle=True)
        )
        client_val_datasets.append(
            make_tf_dataset(x_tr, y_tr, val_idx, batch_size=BATCH_SIZE, shuffle=False)
        )

    test_ds = (
        tf.data.Dataset.from_tensor_slices((x_te, y_te))
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    return client_train_datasets, client_val_datasets, test_ds, client_idxs, client_present, y_tr


client_train_datasets, client_val_datasets, test_ds, client_idxs, client_present, y_train = build_mnist_clients()

print("MNIST client datasets created successfully with fixed seed =", SEED)
for k in range(NUM_CLIENTS):
    missing = sorted(list(set(range(NUM_CLASSES)) - set(client_present[k])))
    print(f"Client {k:02d}: present={client_present[k]} | missing={missing} | n={len(client_idxs[k])}")

# ---------------- Label Distribution Plots ----------------
def plot_client_distribution(y, idxs, title):
    """
    Plot the class distribution for each client.
    """
    cols = 5
    rows = int(np.ceil(len(idxs) / cols))

    plt.figure(figsize=(18, 10))
    for k, idx in enumerate(idxs):
        labels = y[np.array(idx)]
        unique_labels, counts = np.unique(labels, return_counts=True)

        ax = plt.subplot(rows, cols, k + 1)
        ax.bar(unique_labels, counts)
        ax.set_title(f"Client {k}", fontsize=10)
        ax.set_xticks(range(NUM_CLASSES))

    plt.suptitle(title, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


plot_client_distribution(
    y_train,
    client_idxs,
    title=f"MNIST per-client label distribution (each client missing {MISSING_MIN} to {MISSING_MAX} classes)"
)

2: Visualizing the Non-IID Client Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_noniid_heatmap(y, client_idxs, num_clients, num_classes, title):
    """
    Plot a heatmap showing the class distribution across clients.

    Each row corresponds to one client, and each column corresponds
    to one class label. The cell values represent the number of
    samples of each class assigned to each client.
    """
    counts = np.zeros((num_clients, num_classes), dtype=np.int64)

    for client_id in range(num_clients):
        labels = y[np.array(client_idxs[client_id])]
        unique_labels, class_counts = np.unique(labels, return_counts=True)
        counts[client_id, unique_labels] = class_counts

    fig, ax = plt.subplots(figsize=(4, 6))
    im = ax.imshow(counts, aspect="auto")

    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Class label", fontweight="bold")
    ax.set_ylabel("Client ID", fontweight="bold")

    ax.set_xticks(range(num_classes))
    ax.set_yticks(range(num_clients))

    ax.set_xticklabels(range(num_classes), fontweight="bold")
    ax.set_yticklabels(range(num_clients), fontweight="bold")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Number of samples", fontweight="bold")
    for tick in cbar.ax.get_yticklabels():
        tick.set_fontweight("bold")

    plt.tight_layout()
    plt.show()

    return counts


# Generate the class-distribution heatmap
counts = plot_noniid_heatmap(
    y_train,
    client_idxs,
    NUM_CLIENTS,
    NUM_CLASSES,
    f"MNIST label distribution across clients (missing {MISSING_MIN} to {MISSING_MAX} classes)"
)

3: FedAvg Training on MNIST

In [ ]:
# ============================================================
# Purpose:
# - Train a global FedAvg model using the client datasets from Cell 1
# - Track round-level training and validation performance using local metrics
# - Apply early stopping based on mean training accuracy across clients
# - Evaluate on the test set only after training is complete
# - Save the trained global model and metric history
#
# Saves:
#   MNIST_FedAvg_02.h5
#   FedAvg_metrics_rows_02.csv
# ============================================================

import os
import csv
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Training Configuration ----------------
ROUNDS = 20
BATCHES_PER_ROUND = None
LR_BACKBONE = 1e-3

# Early stopping is based only on round-level training accuracy
EARLYSTOP_PATIENCE = 3
EARLYSTOP_MIN_DELTA = 0.0   # Increase slightly if small fluctuations should be ignored

SAVE_DIR = "/content/drive/MyDrive/..."
FEDAVG_SAVE_PATH = f"{SAVE_DIR}/MNIST_FedAvg_02.h5"
FEDAVG_METRICS_CSV = f"{SAVE_DIR}/FedAvg_metrics_rows_02.csv"
os.makedirs(SAVE_DIR, exist_ok=True)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ---------------- Model Definition ----------------
def build_mnist_cnn():
    """
    Build the CNN used as the global model for FedAvg.
    """
    inp = keras.Input(shape=(28, 28, 1))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)

    feat = layers.Dense(128, activation="relu", name="feat")(x)
    logits = layers.Dense(NUM_CLASSES, name="logits")(feat)

    return keras.Model(inp, logits, name="mnist_cnn")


def clone_model_with_weights(model):
    """
    Create a copy of a model, including its current weights.
    """
    cloned = keras.models.clone_model(model)
    cloned.build(model.input_shape)
    cloned.set_weights(model.get_weights())
    return cloned


def average_weights(list_of_weight_lists):
    """
    Compute the element-wise average across client model weights.
    """
    return [np.mean(np.stack(w), axis=0) for w in zip(*list_of_weight_lists)]


def eval_logits(model, ds):
    """
    Evaluate loss and accuracy for a model that outputs logits.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        logits = model(x, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def local_train_full_ce(model, ds, lr=1e-3, batches_limit=None):
    """
    Perform local client training using standard cross-entropy loss.
    Returns the average local training loss and accuracy.
    """
    optimizer = tf.keras.optimizers.Adam(lr)
    acc = tf.keras.metrics.SparseCategoricalAccuracy()

    total_loss, total_n = 0.0, 0
    step_count = 0

    for x, y in ds:
        with tf.GradientTape() as tape:
            logits = model(x, training=True)
            loss = loss_fn(y, logits)

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

        step_count += 1
        if batches_limit is not None and step_count >= batches_limit:
            break

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


# ---------------- Early Stopping ----------------
class TrainAccEarlyStopping:
    """
    Stop training when the monitored training accuracy
    does not improve for a given number of rounds.
    """
    def __init__(self, patience=2, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad = 0

    def step(self, train_acc_value):
        train_acc_value = float(train_acc_value)

        if self.best is None:
            self.best = train_acc_value
            self.bad = 0
            return False

        improved = (train_acc_value - self.best) > self.min_delta
        if improved:
            self.best = train_acc_value
            self.bad = 0
        else:
            self.bad += 1

        return self.bad >= self.patience


stopper = TrainAccEarlyStopping(
    patience=EARLYSTOP_PATIENCE,
    min_delta=EARLYSTOP_MIN_DELTA
)

# ---------------- FedAvg Training ----------------
server = build_mnist_cnn()
_ = server(tf.zeros([1, 28, 28, 1]))

# Metric lists kept explicitly for later export
FedAvg_Training_Loss = []
FedAvg_Training_Acc = []
FedAvg_Val_Loss = []
FedAvg_Val_Acc = []

# Round-level log dictionary
logs = {
    "round": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

for r in range(ROUNDS):
    client_ws = []
    trL, trA, vaL, vaA = [], [], [], []

    for k in range(NUM_CLIENTS):
        local_model = clone_model_with_weights(server)

        local_train_loss, local_train_acc = local_train_full_ce(
            local_model,
            client_train_datasets[k],
            lr=LR_BACKBONE,
            batches_limit=BATCHES_PER_ROUND
        )

        local_val_loss, local_val_acc = eval_logits(
            local_model,
            client_val_datasets[k]
        )

        trL.append(local_train_loss)
        trA.append(local_train_acc)
        vaL.append(local_val_loss)
        vaA.append(local_val_acc)

        client_ws.append(local_model.get_weights())

    # Aggregate client models into the updated global model
    server.set_weights(average_weights(client_ws))

    # Compute mean round-level metrics across clients
    round_tr_loss = float(np.mean(trL))
    round_tr_acc = float(np.mean(trA))
    round_va_loss = float(np.mean(vaL))
    round_va_acc = float(np.mean(vaA))

    # Store metrics for logging and export
    logs["round"].append(r)
    logs["train_loss"].append(round_tr_loss)
    logs["train_acc"].append(round_tr_acc)
    logs["val_loss"].append(round_va_loss)
    logs["val_acc"].append(round_va_acc)

    FedAvg_Training_Loss.append(round_tr_loss)
    FedAvg_Training_Acc.append(round_tr_acc)
    FedAvg_Val_Loss.append(round_va_loss)
    FedAvg_Val_Acc.append(round_va_acc)

    print(
        f"[MNIST-FedAvg][Round {r:02d}] "
        f"Train loss={round_tr_loss:.4f}, acc={round_tr_acc:.4f} | "
        f"Val loss={round_va_loss:.4f}, acc={round_va_acc:.4f}"
    )

    # Stop if training accuracy has not improved
    if stopper.step(round_tr_acc):
        print(
            f"Early stopping at round {r:02d} "
            f"(no improvement in training accuracy for {EARLYSTOP_PATIENCE} rounds)."
        )
        break

# ---------------- Final Test Evaluation ----------------
test_loss, test_acc = eval_logits(server, test_ds)

print("\n[MNIST-FedAvg] Final test evaluation")
print(f"Test loss={test_loss:.4f}, Test acc={test_acc:.4f}")

# ---------------- Save Model ----------------
server.save(FEDAVG_SAVE_PATH)
print("\nSaved MNIST FedAvg global model to:", FEDAVG_SAVE_PATH)

# ---------------- Save Metrics ----------------
rows = [
    ["FedAvg_Training_Loss"] + FedAvg_Training_Loss,
    ["FedAvg_Training_Acc"] + FedAvg_Training_Acc,
    ["FedAvg_Val_Loss"] + FedAvg_Val_Loss,
    ["FedAvg_Val_Acc"] + FedAvg_Val_Acc,
]

with open(FEDAVG_METRICS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print("Saved FedAvg metrics CSV to:", FEDAVG_METRICS_CSV)

4: MHD-FL on MNIST

In [ ]:
# ============================================================
# Purpose:
# - Load the FedAvg model from Cell 3
# - Rebuild the shared backbone and global head
# - Train the backbone, global head, and personal head jointly on each client
# - Aggregate only the shared backbone and global head across clients
# - Keep the personal heads private to each client
# - Track training, validation, and KD-related metrics across rounds
# - Evaluate the final global and personalized models
# - Save the final global model and the combined personalized-head model
# ============================================================

import os
import json
import csv
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Required objects from Cell 1 ----------------
required_vars = [
    "client_train_datasets",
    "client_val_datasets",
    "test_ds",
    "client_present",
    "NUM_CLIENTS",
    "NUM_CLASSES",
    "BATCH_SIZE",
    "SEED",
]
for name in required_vars:
    assert name in globals(), f"Missing {name}. Run Cell 1 first."

SAVE_DIR = "/content/drive/MyDrive/....."
MDL_GLOBAL_PATH = f"{SAVE_DIR}/MDL_global_no_freeze_02.h5"
MDL_HEADS_DIR = f"{SAVE_DIR}/MDL_heads_no_freeze_h5"
MDL_LOGS_PATH = f"{SAVE_DIR}/MDL_logs_no_freeze_02.json"
MDL_METRICS_CSV = f"{SAVE_DIR}/MDL_metrics_no_freeze_rows_02.csv"
os.makedirs(MDL_HEADS_DIR, exist_ok=True)

# ---------------- Locate the FedAvg checkpoint ----------------
fedavg_candidates = (
    glob.glob(f"{SAVE_DIR}/MNIST_FedAvg_02.h5") +
    glob.glob(f"{SAVE_DIR}/*/MNIST_FedAvg_02*.h5") +
    glob.glob(f"{SAVE_DIR}/**/MNIST_FedAvg_02*.h5", recursive=True)
)

fedavg_candidates = sorted(set(fedavg_candidates))

if len(fedavg_candidates) == 0:
    raise FileNotFoundError(f"No FedAvg checkpoint found in {SAVE_DIR}")

FEDAVG_PATH = next(
    (p for p in fedavg_candidates if os.path.basename(p) == "MNIST_FedAvg_02.h5"),
    fedavg_candidates[0]
)
print("Using FedAvg checkpoint:", FEDAVG_PATH)

# ---------------- Training configuration ----------------
ROUNDS = 20
BATCHES_PER_ROUND = None
LR_JOINT = 1e-3

TEMP = 2.0
L_G2P = 1.0
L_P2G = 1.0

EARLYSTOP_PATIENCE = 3
EARLYSTOP_MIN_DELTA = 0.0

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


def load_normalized_mnist_test_like_cell1():
    """
    Reconstruct the MNIST test set using the same preprocessing
    and normalization used in Cell 1.
    """
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()

    y_te = y_te.reshape(-1).astype("int32")
    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7

    x_te = (x_te - mean) / std
    return x_te, y_te


x_test_arr, y_test_arr = load_normalized_mnist_test_like_cell1()


def make_spec_test_ds_for_client(client_id):
    """
    Build a client-specific test set containing only the labels
    available to that client.
    """
    allowed = np.array(sorted(set(client_present[client_id])), dtype=np.int32)
    mask = np.isin(y_test_arr, allowed)

    x = x_test_arr[mask]
    y = y_test_arr[mask]

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, int(mask.sum())


def clone_model_with_weights(model):
    """
    Create a copy of a model together with its current weights.
    """
    cloned = keras.models.clone_model(model)
    cloned.build(model.input_shape)
    cloned.set_weights(model.get_weights())
    return cloned


def average_weights(list_of_weight_lists):
    """
    Compute the element-wise mean across a list of model weights.
    """
    return [np.mean(np.stack(w), axis=0) for w in zip(*list_of_weight_lists)]


def kd_kl(p_logits, q_logits, T=2.0):
    """
    KL divergence between softened prediction distributions.
    """
    p = tf.nn.softmax(p_logits / T, axis=-1)
    q = tf.nn.softmax(q_logits / T, axis=-1)

    kl = tf.reduce_mean(
        tf.reduce_sum(
            p * (tf.math.log(p + 1e-8) - tf.math.log(q + 1e-8)),
            axis=-1
        )
    )
    return kl * (T * T)


def apply_gradients_safe(optimizer, grads, variables):
    """
    Apply gradients after removing any None entries.
    """
    grads_and_vars = [(g, v) for g, v in zip(grads, variables) if g is not None]
    if grads_and_vars:
        optimizer.apply_gradients(grads_and_vars)


def eval_full_model(model, ds):
    """
    Evaluate a full end-to-end model on a dataset.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        logits = model(x, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def eval_head(backbone, head, ds):
    """
    Evaluate a backbone-head pair on a dataset.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        features = backbone(x, training=False)
        logits = head(features, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def local_train_mdl_no_freeze(local_backbone, local_global_head, personal_head, ds,
                              lr=1e-3, batches_limit=None):
    """
    Joint local update for the no-freeze variant.

    The backbone, local global head, and personal head are all updated together.

    Objective:
        CE(global) + CE(personal) + KD(global -> personal) + KD(personal -> global)
    """
    optimizer = tf.keras.optimizers.Adam(lr)

    acc_g = tf.keras.metrics.SparseCategoricalAccuracy()
    acc_p = tf.keras.metrics.SparseCategoricalAccuracy()

    total_ce_g = 0.0
    total_ce_p = 0.0
    total_kd_g2p = 0.0
    total_kd_p2g = 0.0
    total_n = 0
    step_count = 0

    for x, y in ds:
        with tf.GradientTape() as tape:
            features = local_backbone(x, training=True)
            logits_global = local_global_head(features, training=True)
            logits_personal = personal_head(features, training=True)

            ce_global = loss_fn(y, logits_global)
            ce_personal = loss_fn(y, logits_personal)

            kd_g2p = kd_kl(logits_global, logits_personal, T=TEMP)
            kd_p2g = kd_kl(logits_personal, logits_global, T=TEMP)

            loss = (
                ce_global
                + ce_personal
                + (L_G2P * kd_g2p)
                + (L_P2G * kd_p2g)
            )

        variables = (
            local_backbone.trainable_variables
            + local_global_head.trainable_variables
            + personal_head.trainable_variables
        )
        grads = tape.gradient(loss, variables)
        apply_gradients_safe(optimizer, grads, variables)

        bs = int(tf.shape(x)[0].numpy())
        total_ce_g += float(ce_global.numpy()) * bs
        total_ce_p += float(ce_personal.numpy()) * bs
        total_kd_g2p += float(kd_g2p.numpy()) * bs
        total_kd_p2g += float(kd_p2g.numpy()) * bs
        total_n += bs

        acc_g.update_state(y, tf.nn.softmax(logits_global))
        acc_p.update_state(y, tf.nn.softmax(logits_personal))

        step_count += 1
        if batches_limit is not None and step_count >= batches_limit:
            break

    return (
        float(total_ce_g / max(total_n, 1)),
        float(acc_g.result().numpy()),
        float(total_ce_p / max(total_n, 1)),
        float(acc_p.result().numpy()),
        float(total_kd_g2p / max(total_n, 1)),
        float(total_kd_p2g / max(total_n, 1)),
    )


class TrainAccEarlyStopping:
    """
    Stop training when the monitored training accuracy
    does not improve for a fixed number of rounds.
    """
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad_rounds = 0

    def step(self, value):
        value = float(value)

        if self.best is None:
            self.best = value
            self.bad_rounds = 0
            return False

        improved = (value - self.best) > self.min_delta
        if improved:
            self.best = value
            self.bad_rounds = 0
        else:
            self.bad_rounds += 1

        return self.bad_rounds >= self.patience


stopper = TrainAccEarlyStopping(
    patience=EARLYSTOP_PATIENCE,
    min_delta=EARLYSTOP_MIN_DELTA
)

# ---------------- Load the FedAvg model ----------------
fedavg_model = keras.models.load_model(FEDAVG_PATH, compile=False)
print("Loaded FedAvg model.")

# Reconstruct the shared backbone and global head from the FedAvg checkpoint
backbone = keras.Model(
    fedavg_model.input,
    fedavg_model.get_layer("feat").output,
    name="backbone"
)
feat_dim = backbone.output_shape[-1]

server_global_head = layers.Dense(NUM_CLASSES, name="server_global_head")
server_global_head(tf.zeros([1, feat_dim]))
server_global_head.set_weights(fedavg_model.get_layer("logits").get_weights())

# Confirm that the reconstructed backbone/head path matches the original model
init_loss_full, init_acc_full = eval_full_model(fedavg_model, test_ds)
init_loss_split, init_acc_split = eval_head(backbone, server_global_head, test_ds)

print("\nInitial consistency check")
print(f"Full FedAvg model:       loss={init_loss_full:.4f}, acc={init_acc_full:.4f}")
print(f"Backbone + global head:  loss={init_loss_split:.4f}, acc={init_acc_split:.4f}")

if abs(init_loss_full - init_loss_split) > 1e-5 or abs(init_acc_full - init_acc_split) > 1e-5:
    raise ValueError(
        "Mismatch between the loaded FedAvg model and the reconstructed backbone/head path."
    )

# ---------------- Initialize client-specific heads ----------------
personal_heads = []
for client_id in range(NUM_CLIENTS):
    head = layers.Dense(NUM_CLASSES, name=f"personal_head_{client_id}")
    head(tf.zeros([1, feat_dim]))
    head.set_weights(server_global_head.get_weights())
    personal_heads.append(head)

# Round-level metric storage
logs = {
    "round": [],
    "train_loss_global": [],
    "train_acc_global": [],
    "train_loss_personal": [],
    "train_acc_personal": [],
    "val_loss_global": [],
    "val_acc_global": [],
    "val_loss_personal": [],
    "val_acc_personal": [],
    "kd_g2p": [],
    "kd_p2g": [],
}

# ---------------- Federated training loop ----------------
for r in range(ROUNDS):
    client_backbone_ws = []
    client_global_ws = []

    trLg, trAg = [], []
    trLp, trAp = [], []
    vaLg, vaAg = [], []
    vaLp, vaAp = [], []
    kd_g2p_vals, kd_p2g_vals = [], []

    for client_id in range(NUM_CLIENTS):
        local_backbone = clone_model_with_weights(backbone)

        local_global_head = layers.Dense(NUM_CLASSES, name=f"local_global_head_{client_id}")
        local_global_head(tf.zeros([1, feat_dim]))
        local_global_head.set_weights(server_global_head.get_weights())

        l_g, a_g, l_p, a_p, kd_g2p, kd_p2g = local_train_mdl_no_freeze(
            local_backbone=local_backbone,
            local_global_head=local_global_head,
            personal_head=personal_heads[client_id],
            ds=client_train_datasets[client_id],
            lr=LR_JOINT,
            batches_limit=BATCHES_PER_ROUND,
        )

        lvg, avg = eval_head(local_backbone, local_global_head, client_val_datasets[client_id])
        lvp, avp = eval_head(local_backbone, personal_heads[client_id], client_val_datasets[client_id])

        trLg.append(l_g)
        trAg.append(a_g)
        trLp.append(l_p)
        trAp.append(a_p)

        vaLg.append(lvg)
        vaAg.append(avg)
        vaLp.append(lvp)
        vaAp.append(avp)

        kd_g2p_vals.append(kd_g2p)
        kd_p2g_vals.append(kd_p2g)

        client_backbone_ws.append(local_backbone.get_weights())
        client_global_ws.append(local_global_head.get_weights())

    # Aggregate only the shared components
    backbone.set_weights(average_weights(client_backbone_ws))
    server_global_head.set_weights(average_weights(client_global_ws))

    round_tr_lg = float(np.mean(trLg))
    round_tr_ag = float(np.mean(trAg))
    round_tr_lp = float(np.mean(trLp))
    round_tr_ap = float(np.mean(trAp))

    round_va_lg = float(np.mean(vaLg))
    round_va_ag = float(np.mean(vaAg))
    round_va_lp = float(np.mean(vaLp))
    round_va_ap = float(np.mean(vaAp))

    round_kd_g2p = float(np.mean(kd_g2p_vals))
    round_kd_p2g = float(np.mean(kd_p2g_vals))

    logs["round"].append(r)
    logs["train_loss_global"].append(round_tr_lg)
    logs["train_acc_global"].append(round_tr_ag)
    logs["train_loss_personal"].append(round_tr_lp)
    logs["train_acc_personal"].append(round_tr_ap)
    logs["val_loss_global"].append(round_va_lg)
    logs["val_acc_global"].append(round_va_ag)
    logs["val_loss_personal"].append(round_va_lp)
    logs["val_acc_personal"].append(round_va_ap)
    logs["kd_g2p"].append(round_kd_g2p)
    logs["kd_p2g"].append(round_kd_p2g)

    print(
        f"[MDL no-freeze][Round {r:02d}] "
        f"G train: loss={round_tr_lg:.3f}, acc={round_tr_ag:.3f} | "
        f"P train: loss={round_tr_lp:.3f}, acc={round_tr_ap:.3f} || "
        f"G val: loss={round_va_lg:.3f}, acc={round_va_ag:.3f} | "
        f"P val: loss={round_va_lp:.3f}, acc={round_va_ap:.3f} | "
        f"KD g->p={round_kd_g2p:.4f}, p->g={round_kd_p2g:.4f}"
    )

    if stopper.step(round_tr_ap):
        print(
            f"Stopping early at round {r:02d} "
            f"(no improvement in personal training accuracy for {EARLYSTOP_PATIENCE} rounds)."
        )
        break

# ---------------- Final evaluation ----------------
global_test_loss, global_test_acc = eval_head(backbone, server_global_head, test_ds)

spec_losses, spec_accs = [], []
glob_losses, glob_accs = [], []

for client_id in range(NUM_CLIENTS):
    spec_ds_k, _ = make_spec_test_ds_for_client(client_id)

    l_spec, a_spec = eval_head(backbone, personal_heads[client_id], spec_ds_k)
    l_glob, a_glob = eval_head(backbone, personal_heads[client_id], test_ds)

    spec_losses.append(l_spec)
    spec_accs.append(a_spec)
    glob_losses.append(l_glob)
    glob_accs.append(a_glob)

print("\nFinal results")
print(f"Global model on full test set: loss={global_test_loss:.4f}, acc={global_test_acc:.4f}")
print(f"Average personalized model on specialization test: loss={float(np.mean(spec_losses)):.4f}, acc={float(np.mean(spec_accs)):.4f}")
print(f"Average personalized model on full test set: loss={float(np.mean(glob_losses)):.4f}, acc={float(np.mean(glob_accs)):.4f}")

# ---------------- Save the final models ----------------
inp = keras.Input(shape=fedavg_model.input_shape[1:])
out = server_global_head(backbone(inp))

global_model = keras.Model(inp, out, name="global_HD_FL")
GLOBAL_MODEL_PATH = os.path.join(SAVE_DIR, "global_HD-FL.h5")
global_model.save(GLOBAL_MODEL_PATH)

print("\nSaved global model to:", GLOBAL_MODEL_PATH)

inputs = keras.Input(shape=(feat_dim,))
outputs = []
for head in personal_heads:
    outputs.append(head(inputs))

personal_model = keras.Model(inputs, outputs, name="personal_HD_FL")
PERSONAL_MODEL_PATH = os.path.join(SAVE_DIR, "person_HD-FL.h5")
personal_model.save(PERSONAL_MODEL_PATH)

print("Saved personalized heads model to:", PERSONAL_MODEL_PATH)

Ablation Study 1 — Strict Decoupling

In [ ]:
# Purpose:
# - Initialize from the FedAvg checkpoint saved in Cell 2
# - Keep one shared backbone and one shared global head on the server
# - Maintain one private personal head per client across rounds
# - Update the backbone using only the global head loss
# - Update the global and personal heads using cross-entropy and mutual KD
# - Aggregate only the backbone and global head across clients
# - Apply early stopping based on average personalized training accuracy
# - Track round-level global/personal train and validation metrics, along with KD terms
# - Evaluate:
#     (1) the final global model on the full test set
#     (2) the final personalized heads on their specialization test sets
#     (3) the final personalized heads on the full test set
# - Save the final global model, personalized heads, logs, and CSV metrics
#
# Saves:
#   MDL_global.h5
#   MDL_heads_h5/
#   MDL_logs.json
#   MDL_metrics_rows.csv
# ============================================================

import os
import json
import glob
import csv
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Required objects from Cell 1 ----------------
assert "client_train_datasets" in globals()
assert "client_val_datasets" in globals()
assert "test_ds" in globals()
assert "client_present" in globals()
assert "NUM_CLIENTS" in globals()
assert "NUM_CLASSES" in globals()
assert "BATCH_SIZE" in globals()
assert "SEED" in globals()

SAVE_DIR = "/content/drive/MyDrive/..."
MDL_GLOBAL_PATH = f"{SAVE_DIR}/MDL_global.h5"
MDL_HEADS_DIR = f"{SAVE_DIR}/MDL_heads_h5"
MDL_LOGS_PATH = f"{SAVE_DIR}/MDL_logs.json"
MDL_METRICS_CSV = f"{SAVE_DIR}/MDL_metrics_rows.csv"
os.makedirs(MDL_HEADS_DIR, exist_ok=True)

# ---------------- Locate the FedAvg checkpoint ----------------
fedavg_candidates = (
    glob.glob(f"{SAVE_DIR}/MNIST_FedAvg_02.h5") +
    glob.glob(f"{SAVE_DIR}/*MNIST_FedAvg_02*.h5") +
    glob.glob(f"{SAVE_DIR}/**/MNIST_FedAvg_02*.h5", recursive=True)
)
fedavg_candidates = sorted(set(fedavg_candidates))

print("FedAvg candidates:", fedavg_candidates)

if len(fedavg_candidates) == 0:
    raise FileNotFoundError(
        f"No FedAvg checkpoint found in {SAVE_DIR}. "
        f"Current contents: {os.listdir(SAVE_DIR)}"
    )

FEDAVG_PATH = fedavg_candidates[0]
print("Using FedAvg checkpoint:", FEDAVG_PATH)

# ---------------- Training configuration ----------------
ROUNDS = 20
BATCHES_PER_ROUND = None
LR_BACKBONE = 1e-3
LR_HEAD = 2e-3

TEMP = 2.0
L_G2P = 1.0
L_P2G = 0.1

# Early stopping is based on round-level personalized training accuracy
EARLYSTOP_PATIENCE = 3
EARLYSTOP_MIN_DELTA = 0.0

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# ---------------- Rebuild the normalized MNIST test arrays ----------------
# This follows the same preprocessing used in Cell 1 so that
# specialization test sets are constructed consistently.
def load_normalized_mnist_arrays_like_cell1():
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()

    y_tr = y_tr.reshape(-1).astype("int32")
    y_te = y_te.reshape(-1).astype("int32")

    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7

    x_tr = (x_tr - mean) / std
    x_te = (x_te - mean) / std
    return x_te, y_te


x_test_arr, y_test_arr = load_normalized_mnist_arrays_like_cell1()


def make_spec_test_ds_for_client(client_id):
    """
    Build a client-specific test set containing only the labels
    available to that client.
    """
    allowed = np.array(sorted(list(set(client_present[client_id]))), dtype=np.int32)
    mask = np.isin(y_test_arr, allowed)

    x = x_test_arr[mask]
    y = y_test_arr[mask]

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, int(mask.sum())


# ---------------- Helper functions ----------------
def clone_model_with_weights(model):
    """
    Create a copy of a model together with its current weights.
    """
    cloned = keras.models.clone_model(model)
    cloned.build(model.input_shape)
    cloned.set_weights(model.get_weights())
    return cloned


def average_weights(list_of_weight_lists):
    """
    Compute the element-wise mean across a list of model weights.
    """
    return [np.mean(np.stack(w), axis=0) for w in zip(*list_of_weight_lists)]


def kd_kl(p_logits, q_logits, T=2.0):
    """
    Compute the KL divergence between two softened prediction distributions.
    """
    p = tf.nn.softmax(p_logits / T, axis=-1)
    q = tf.nn.softmax(q_logits / T, axis=-1)

    kl = tf.reduce_mean(
        tf.reduce_sum(
            p * (tf.math.log(p + 1e-8) - tf.math.log(q + 1e-8)),
            axis=-1
        )
    )
    return kl * (T * T)


def eval_head(backbone, head, ds):
    """
    Evaluate a backbone-head pair on a dataset.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        features = backbone(x, training=False)
        logits = head(features, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


# ---------------- Local training for strict decoupling ----------------
def local_train_head_only_mdl(local_backbone, local_global_head, personal_head, ds,
                              lr_backbone=1e-3, lr_head=2e-3, batches_limit=None):
    """
    Local training for the strict decoupling variant.

    Step 1:
        Update the backbone using only the global-head cross-entropy loss.

    Step 2:
        Freeze the features, then update the global head and personal head
        using cross-entropy and mutual knowledge distillation.
    """
    optimizer_backbone = tf.keras.optimizers.Adam(lr_backbone)
    optimizer_head = tf.keras.optimizers.Adam(lr_head)

    acc_global = tf.keras.metrics.SparseCategoricalAccuracy()
    acc_personal = tf.keras.metrics.SparseCategoricalAccuracy()

    total_ce_global = 0.0
    total_ce_personal = 0.0
    total_kd_g2p = 0.0
    total_kd_p2g = 0.0
    total_n = 0
    step_count = 0

    for x, y in ds:
        # Update the backbone using only the global-head classification loss
        with tf.GradientTape() as tape_backbone:
            features = local_backbone(x, training=True)
            logits_global_backbone = local_global_head(features, training=True)
            loss_backbone = loss_fn(y, logits_global_backbone)

        grads_backbone = tape_backbone.gradient(loss_backbone, local_backbone.trainable_variables)
        optimizer_backbone.apply_gradients(zip(grads_backbone, local_backbone.trainable_variables))

        # Freeze features and update the two heads with CE + mutual KD
        frozen_features = tf.stop_gradient(local_backbone(x, training=False))

        with tf.GradientTape() as tape_head:
            logits_global = local_global_head(frozen_features, training=True)
            logits_personal = personal_head(frozen_features, training=True)

            ce_global = loss_fn(y, logits_global)
            ce_personal = loss_fn(y, logits_personal)

            kd_g2p = kd_kl(logits_global, logits_personal, T=TEMP)
            kd_p2g = kd_kl(logits_personal, logits_global, T=TEMP)

            loss_heads = (
                ce_global
                + ce_personal
                + (L_G2P * kd_g2p)
                + (L_P2G * kd_p2g)
            )

        head_vars = local_global_head.trainable_variables + personal_head.trainable_variables
        grads_head = tape_head.gradient(loss_heads, head_vars)
        optimizer_head.apply_gradients(zip(grads_head, head_vars))

        bs = int(tf.shape(x)[0].numpy())
        total_ce_global += float(ce_global.numpy()) * bs
        total_ce_personal += float(ce_personal.numpy()) * bs
        total_kd_g2p += float(kd_g2p.numpy()) * bs
        total_kd_p2g += float(kd_p2g.numpy()) * bs
        total_n += bs

        acc_global.update_state(y, tf.nn.softmax(logits_global))
        acc_personal.update_state(y, tf.nn.softmax(logits_personal))

        step_count += 1
        if batches_limit is not None and step_count >= batches_limit:
            break

    return (
        float(total_ce_global / max(total_n, 1)),
        float(acc_global.result().numpy()),
        float(total_ce_personal / max(total_n, 1)),
        float(acc_personal.result().numpy()),
        float(total_kd_g2p / max(total_n, 1)),
        float(total_kd_p2g / max(total_n, 1)),
    )


# ---------------- Early stopping ----------------
class TrainAccEarlyStopping:
    """
    Stop training when the monitored personalized training accuracy
    does not improve for a fixed number of rounds.
    """
    def __init__(self, patience=2, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad = 0

    def step(self, value):
        value = float(value)

        if self.best is None:
            self.best = value
            self.bad = 0
            return False

        improved = (value - self.best) > self.min_delta
        if improved:
            self.best = value
            self.bad = 0
        else:
            self.bad += 1

        return self.bad >= self.patience


stopper = TrainAccEarlyStopping(
    patience=EARLYSTOP_PATIENCE,
    min_delta=EARLYSTOP_MIN_DELTA
)

# ---------------- Load the FedAvg model and initialize shared components ----------------
fedavg_model = keras.models.load_model(FEDAVG_PATH)
print("Loaded FedAvg model.")

backbone = keras.Model(
    fedavg_model.input,
    fedavg_model.get_layer("feat").output,
    name="backbone"
)
feat_dim = backbone.output_shape[-1]

server_global_head = layers.Dense(NUM_CLASSES, name="server_global_head")
server_global_head(tf.zeros([1, feat_dim]))
server_global_head.set_weights(fedavg_model.get_layer("logits").get_weights())

# Initialize one private head per client from the same global head
personal_heads = []
for client_id in range(NUM_CLIENTS):
    head = layers.Dense(NUM_CLASSES, name=f"personal_head_{client_id}")
    head(tf.zeros([1, feat_dim]))
    head.set_weights(server_global_head.get_weights())
    personal_heads.append(head)

# Metric storage
logs = {
    "round": [],
    "train_loss_global": [],
    "train_acc_global": [],
    "train_loss_personal": [],
    "train_acc_personal": [],
    "val_loss_global": [],
    "val_acc_global": [],
    "val_loss_personal": [],
    "val_acc_personal": [],
    "kd_g2p": [],
    "kd_p2g": [],
}

# ---------------- Federated training loop ----------------
for r in range(ROUNDS):
    client_backbone_ws = []
    client_global_ws = []

    train_loss_global_vals, train_acc_global_vals = [], []
    train_loss_personal_vals, train_acc_personal_vals = [], []
    val_loss_global_vals, val_acc_global_vals = [], []
    val_loss_personal_vals, val_acc_personal_vals = [], []
    kd_g2p_vals, kd_p2g_vals = [], []

    for client_id in range(NUM_CLIENTS):
        local_backbone = clone_model_with_weights(backbone)

        local_global_head = layers.Dense(NUM_CLASSES, name=f"local_global_head_{client_id}")
        local_global_head(tf.zeros([1, feat_dim]))
        local_global_head.set_weights(server_global_head.get_weights())

        train_loss_global, train_acc_global, train_loss_personal, train_acc_personal, kd_g2p, kd_p2g = (
            local_train_head_only_mdl(
                local_backbone,
                local_global_head,
                personal_heads[client_id],
                client_train_datasets[client_id],
                lr_backbone=LR_BACKBONE,
                lr_head=LR_HEAD,
                batches_limit=BATCHES_PER_ROUND,
            )
        )

        val_loss_global, val_acc_global = eval_head(
            local_backbone, local_global_head, client_val_datasets[client_id]
        )
        val_loss_personal, val_acc_personal = eval_head(
            local_backbone, personal_heads[client_id], client_val_datasets[client_id]
        )

        train_loss_global_vals.append(train_loss_global)
        train_acc_global_vals.append(train_acc_global)
        train_loss_personal_vals.append(train_loss_personal)
        train_acc_personal_vals.append(train_acc_personal)

        val_loss_global_vals.append(val_loss_global)
        val_acc_global_vals.append(val_acc_global)
        val_loss_personal_vals.append(val_loss_personal)
        val_acc_personal_vals.append(val_acc_personal)

        kd_g2p_vals.append(kd_g2p)
        kd_p2g_vals.append(kd_p2g)

        client_backbone_ws.append(local_backbone.get_weights())
        client_global_ws.append(local_global_head.get_weights())

    # Aggregate the shared backbone and shared global head
    backbone.set_weights(average_weights(client_backbone_ws))
    server_global_head.set_weights(average_weights(client_global_ws))

    round_train_loss_global = float(np.mean(train_loss_global_vals))
    round_train_acc_global = float(np.mean(train_acc_global_vals))
    round_train_loss_personal = float(np.mean(train_loss_personal_vals))
    round_train_acc_personal = float(np.mean(train_acc_personal_vals))

    round_val_loss_global = float(np.mean(val_loss_global_vals))
    round_val_acc_global = float(np.mean(val_acc_global_vals))
    round_val_loss_personal = float(np.mean(val_loss_personal_vals))
    round_val_acc_personal = float(np.mean(val_acc_personal_vals))

    round_kd_g2p = float(np.mean(kd_g2p_vals))
    round_kd_p2g = float(np.mean(kd_p2g_vals))

    logs["round"].append(r)
    logs["train_loss_global"].append(round_train_loss_global)
    logs["train_acc_global"].append(round_train_acc_global)
    logs["train_loss_personal"].append(round_train_loss_personal)
    logs["train_acc_personal"].append(round_train_acc_personal)
    logs["val_loss_global"].append(round_val_loss_global)
    logs["val_acc_global"].append(round_val_acc_global)
    logs["val_loss_personal"].append(round_val_loss_personal)
    logs["val_acc_personal"].append(round_val_acc_personal)
    logs["kd_g2p"].append(round_kd_g2p)
    logs["kd_p2g"].append(round_kd_p2g)

    print(
        f"[Strict Decoupling][Round {r:02d}] "
        f"G(train) loss={round_train_loss_global:.3f} acc={round_train_acc_global:.3f} | "
        f"P(train) loss={round_train_loss_personal:.3f} acc={round_train_acc_personal:.3f} || "
        f"G(val) loss={round_val_loss_global:.3f} acc={round_val_acc_global:.3f} | "
        f"P(val) loss={round_val_loss_personal:.3f} acc={round_val_acc_personal:.3f} | "
        f"KD g->p={round_kd_g2p:.4f}, p->g={round_kd_p2g:.4f}"
    )

    if stopper.step(round_train_acc_personal):
        print(
            f"Stopping early at round {r:02d} "
            f"(no improvement in personalized training accuracy for {EARLYSTOP_PATIENCE} rounds)."
        )
        break

# ---------------- Final evaluation ----------------
global_test_loss, global_test_acc = eval_head(backbone, server_global_head, test_ds)

spec_losses, spec_accs = [], []
glob_losses, glob_accs = [], []

for client_id in range(NUM_CLIENTS):
    spec_ds_k, _ = make_spec_test_ds_for_client(client_id)

    spec_loss, spec_acc = eval_head(backbone, personal_heads[client_id], spec_ds_k)
    glob_loss, glob_acc = eval_head(backbone, personal_heads[client_id], test_ds)

    spec_losses.append(spec_loss)
    spec_accs.append(spec_acc)
    glob_losses.append(glob_loss)
    glob_accs.append(glob_acc)

print("\nFinal results")
print(f"Global model on full test set: loss={global_test_loss:.4f}, acc={global_test_acc:.4f}")
print(f"Average personalized model on specialization test: loss={float(np.mean(spec_losses)):.4f}, acc={float(np.mean(spec_accs)):.4f}")
print(f"Average personalized model on full test set: loss={float(np.mean(glob_losses)):.4f}, acc={float(np.mean(glob_accs)):.4f}")

# ---------------- Save models and logs ----------------
inp = keras.Input(shape=fedavg_model.input_shape[1:])
out = server_global_head(backbone(inp))
mdl_global_model = keras.Model(inp, out, name="MDL_global_model")
mdl_global_model.save(MDL_GLOBAL_PATH)
print("\nSaved MDL global model to:", MDL_GLOBAL_PATH)

for client_id in range(NUM_CLIENTS):
    head_input = keras.Input(shape=(feat_dim,))
    head_output = personal_heads[client_id](head_input)
    head_model = keras.Model(head_input, head_output, name=f"personal_head_{client_id}")
    head_model.save(f"{MDL_HEADS_DIR}/personal_head_{client_id}.h5")
print("Saved MDL personal heads to:", MDL_HEADS_DIR)

with open(MDL_LOGS_PATH, "w") as f:
    json.dump(logs, f, indent=2)
print("Saved MDL logs to:", MDL_LOGS_PATH)

rows = [
    ["round"] + logs["round"],
    ["train_loss_global"] + logs["train_loss_global"],
    ["train_acc_global"] + logs["train_acc_global"],
    ["train_loss_personal"] + logs["train_loss_personal"],
    ["train_acc_personal"] + logs["train_acc_personal"],
    ["val_loss_global"] + logs["val_loss_global"],
    ["val_acc_global"] + logs["val_acc_global"],
    ["val_loss_personal"] + logs["val_loss_personal"],
    ["val_acc_personal"] + logs["val_acc_personal"],
    ["kd_g2p"] + logs["kd_g2p"],
    ["kd_p2g"] + logs["kd_p2g"],
]

with open(MDL_METRICS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print("Saved MDL metrics CSV to:", MDL_METRICS_CSV)

Ablation Study 2 — G→P Only KD.

In [ ]:
# ===========================================================
# Mutual Distillation on MNIST with Global-to-Personal Transfer Only
#
# Purpose:
# - Initialize from the FedAvg checkpoint saved in Cell 2
# - Rebuild the shared backbone and shared global head
# - Maintain one private personal head per client
# - Train the backbone, global head, and personal head jointly on each client
# - Use only the global-to-personal KD direction
# - Aggregate only the shared backbone and global head across clients
# - Apply early stopping based on average personalized training accuracy
# - Track round-level global/personal training and validation metrics, along with G→P KD
# - Evaluate:
#     (1) the final global model on the full test set
#     (2) the final personalized models on their specialization test sets
#     (3) the final personalized models on the full test set
# - Save the final models, logs, and CSV metrics
#
# Saves:
#   MDL_global_g2p_only_02.h5
#   MDL_heads_g2p_only_h5/
#   MDL_logs_g2p_only_02.json
#   MDL_metrics_g2p_only_rows_02.csv
# ============================================================

import os
import json
import csv
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Required objects from Cell 1 ----------------
required_vars = [
    "client_train_datasets",
    "client_val_datasets",
    "test_ds",
    "client_present",
    "NUM_CLIENTS",
    "NUM_CLASSES",
    "BATCH_SIZE",
    "SEED",
]
for name in required_vars:
    assert name in globals(), f"Missing {name}. Run Cell 1 first."

SAVE_DIR = "/content/drive/MyDrive/...."
MDL_GLOBAL_PATH = f"{SAVE_DIR}/MDL_global_g2p_only_02.h5"
MDL_HEADS_DIR = f"{SAVE_DIR}/MDL_heads_g2p_only_h5"
MDL_LOGS_PATH = f"{SAVE_DIR}/MDL_logs_g2p_only_02.json"
MDL_METRICS_CSV = f"{SAVE_DIR}/MDL_metrics_g2p_only_rows_02.csv"
os.makedirs(MDL_HEADS_DIR, exist_ok=True)

# ---------------- Locate the FedAvg checkpoint ----------------
fedavg_candidates = (
    glob.glob(f"{SAVE_DIR}/MNIST_FedAvg_02.h5") +
    glob.glob(f"{SAVE_DIR}/*/MNIST_FedAvg_02*.h5") +
    glob.glob(f"{SAVE_DIR}/**/MNIST_FedAvg_02*.h5", recursive=True)
)
fedavg_candidates = sorted(set(fedavg_candidates))

if len(fedavg_candidates) == 0:
    raise FileNotFoundError(f"No FedAvg checkpoint found in {SAVE_DIR}")

FEDAVG_PATH = next(
    (p for p in fedavg_candidates if os.path.basename(p) == "MNIST_FedAvg_02.h5"),
    fedavg_candidates[0]
)
print("Using FedAvg checkpoint:", FEDAVG_PATH)

# ---------------- Training configuration ----------------
ROUNDS = 20
BATCHES_PER_ROUND = None
LR_JOINT = 1e-3

TEMP = 2.0
L_G2P = 1.0

EARLYSTOP_PATIENCE = 3
EARLYSTOP_MIN_DELTA = 0.0

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


def load_normalized_mnist_test_like_cell1():
    """
    Rebuild the MNIST test set using the same preprocessing
    and normalization used in Cell 1.
    """
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()

    y_te = y_te.reshape(-1).astype("int32")
    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7

    x_te = (x_te - mean) / std
    return x_te, y_te


x_test_arr, y_test_arr = load_normalized_mnist_test_like_cell1()


def make_spec_test_ds_for_client(client_id):
    """
    Build a client-specific test set containing only the labels
    available to that client.
    """
    allowed = np.array(sorted(set(client_present[client_id])), dtype=np.int32)
    mask = np.isin(y_test_arr, allowed)

    x = x_test_arr[mask]
    y = y_test_arr[mask]

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, int(mask.sum())


def clone_model_with_weights(model):
    """
    Create a copy of a model together with its current weights.
    """
    cloned = keras.models.clone_model(model)
    cloned.build(model.input_shape)
    cloned.set_weights(model.get_weights())
    return cloned


def average_weights(list_of_weight_lists):
    """
    Compute the element-wise mean across a list of model weights.
    """
    return [np.mean(np.stack(w), axis=0) for w in zip(*list_of_weight_lists)]


def kd_kl(p_logits, q_logits, T=2.0):
    """
    Compute KL(softmax(p/T) || softmax(q/T)) scaled by T^2.
    """
    p = tf.nn.softmax(p_logits / T, axis=-1)
    q = tf.nn.softmax(q_logits / T, axis=-1)

    kl = tf.reduce_mean(
        tf.reduce_sum(
            p * (tf.math.log(p + 1e-8) - tf.math.log(q + 1e-8)),
            axis=-1
        )
    )
    return kl * (T * T)


def apply_gradients_safe(optimizer, grads, variables):
    """
    Apply gradients after removing any None entries.
    """
    grads_and_vars = [(g, v) for g, v in zip(grads, variables) if g is not None]
    if grads_and_vars:
        optimizer.apply_gradients(grads_and_vars)


def eval_full_model(model, ds):
    """
    Evaluate a full end-to-end model on a dataset.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        logits = model(x, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def eval_head(backbone, head, ds):
    """
    Evaluate a backbone-head pair on a dataset.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        features = backbone(x, training=False)
        logits = head(features, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def local_train_mdl_g2p_only(local_backbone, local_global_head, personal_head, ds,
                             lr=1e-3, batches_limit=None):
    """
    Joint local update with only the global-to-personal KD direction.

    Updated components:
      - backbone
      - local global head
      - personal head

    Objective:
      CE(global) + CE(personal) + KD(global -> personal)
    """
    optimizer = tf.keras.optimizers.Adam(lr)

    acc_g = tf.keras.metrics.SparseCategoricalAccuracy()
    acc_p = tf.keras.metrics.SparseCategoricalAccuracy()

    total_ce_g = 0.0
    total_ce_p = 0.0
    total_kd_g2p = 0.0
    total_n = 0
    step_count = 0

    for x, y in ds:
        with tf.GradientTape() as tape:
            features = local_backbone(x, training=True)
            logits_global = local_global_head(features, training=True)
            logits_personal = personal_head(features, training=True)

            ce_global = loss_fn(y, logits_global)
            ce_personal = loss_fn(y, logits_personal)

            kd_g2p = kd_kl(logits_global, logits_personal, T=TEMP)

            loss = ce_global + ce_personal + (L_G2P * kd_g2p)

        variables = (
            local_backbone.trainable_variables
            + local_global_head.trainable_variables
            + personal_head.trainable_variables
        )
        grads = tape.gradient(loss, variables)
        apply_gradients_safe(optimizer, grads, variables)

        bs = int(tf.shape(x)[0].numpy())
        total_ce_g += float(ce_global.numpy()) * bs
        total_ce_p += float(ce_personal.numpy()) * bs
        total_kd_g2p += float(kd_g2p.numpy()) * bs
        total_n += bs

        acc_g.update_state(y, tf.nn.softmax(logits_global))
        acc_p.update_state(y, tf.nn.softmax(logits_personal))

        step_count += 1
        if batches_limit is not None and step_count >= batches_limit:
            break

    return (
        float(total_ce_g / max(total_n, 1)),
        float(acc_g.result().numpy()),
        float(total_ce_p / max(total_n, 1)),
        float(acc_p.result().numpy()),
        float(total_kd_g2p / max(total_n, 1)),
    )


class TrainAccEarlyStopping:
    """
    Stop training when the monitored personalized training accuracy
    does not improve for a fixed number of rounds.
    """
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad_rounds = 0

    def step(self, value):
        value = float(value)

        if self.best is None:
            self.best = value
            self.bad_rounds = 0
            return False

        improved = (value - self.best) > self.min_delta
        if improved:
            self.best = value
            self.bad_rounds = 0
        else:
            self.bad_rounds += 1

        return self.bad_rounds >= self.patience


stopper = TrainAccEarlyStopping(
    patience=EARLYSTOP_PATIENCE,
    min_delta=EARLYSTOP_MIN_DELTA
)

# ---------------- Load the FedAvg model ----------------
fedavg_model = keras.models.load_model(FEDAVG_PATH, compile=False)
print("Loaded FedAvg model.")

# Reconstruct the shared backbone and global head
backbone = keras.Model(
    fedavg_model.input,
    fedavg_model.get_layer("feat").output,
    name="backbone"
)
feat_dim = backbone.output_shape[-1]

server_global_head = layers.Dense(NUM_CLASSES, name="server_global_head")
server_global_head(tf.zeros([1, feat_dim]))
server_global_head.set_weights(fedavg_model.get_layer("logits").get_weights())

# Confirm that the reconstructed backbone/head path matches the full model
init_loss_full, init_acc_full = eval_full_model(fedavg_model, test_ds)
init_loss_split, init_acc_split = eval_head(backbone, server_global_head, test_ds)

print("\nInitial consistency check")
print(f"Full FedAvg model:       loss={init_loss_full:.4f}, acc={init_acc_full:.4f}")
print(f"Backbone + global head:  loss={init_loss_split:.4f}, acc={init_acc_split:.4f}")

if abs(init_loss_full - init_loss_split) > 1e-5 or abs(init_acc_full - init_acc_split) > 1e-5:
    raise ValueError(
        "Mismatch between the loaded FedAvg model and the reconstructed backbone/head path."
    )

# Initialize one private head per client
personal_heads = []
for client_id in range(NUM_CLIENTS):
    head = layers.Dense(NUM_CLASSES, name=f"personal_head_{client_id}")
    head(tf.zeros([1, feat_dim]))
    head.set_weights(server_global_head.get_weights())
    personal_heads.append(head)

# Metric storage
logs = {
    "round": [],
    "train_loss_global": [],
    "train_acc_global": [],
    "train_loss_personal": [],
    "train_acc_personal": [],
    "val_loss_global": [],
    "val_acc_global": [],
    "val_loss_personal": [],
    "val_acc_personal": [],
    "kd_g2p": [],
}

# ---------------- Federated training loop ----------------
for r in range(ROUNDS):
    client_backbone_ws = []
    client_global_ws = []

    trLg, trAg = [], []
    trLp, trAp = [], []
    vaLg, vaAg = [], []
    vaLp, vaAp = [], []
    kd_g2p_vals = []

    for client_id in range(NUM_CLIENTS):
        local_backbone = clone_model_with_weights(backbone)

        local_global_head = layers.Dense(NUM_CLASSES, name=f"local_global_head_{client_id}")
        local_global_head(tf.zeros([1, feat_dim]))
        local_global_head.set_weights(server_global_head.get_weights())

        l_g, a_g, l_p, a_p, kd_g2p = local_train_mdl_g2p_only(
            local_backbone=local_backbone,
            local_global_head=local_global_head,
            personal_head=personal_heads[client_id],
            ds=client_train_datasets[client_id],
            lr=LR_JOINT,
            batches_limit=BATCHES_PER_ROUND
        )

        lvg, avg = eval_head(local_backbone, local_global_head, client_val_datasets[client_id])
        lvp, avp = eval_head(local_backbone, personal_heads[client_id], client_val_datasets[client_id])

        trLg.append(l_g)
        trAg.append(a_g)
        trLp.append(l_p)
        trAp.append(a_p)

        vaLg.append(lvg)
        vaAg.append(avg)
        vaLp.append(lvp)
        vaAp.append(avp)

        kd_g2p_vals.append(kd_g2p)

        client_backbone_ws.append(local_backbone.get_weights())
        client_global_ws.append(local_global_head.get_weights())

    # Aggregate only the shared components
    backbone.set_weights(average_weights(client_backbone_ws))
    server_global_head.set_weights(average_weights(client_global_ws))

    round_tr_lg = float(np.mean(trLg))
    round_tr_ag = float(np.mean(trAg))
    round_tr_lp = float(np.mean(trLp))
    round_tr_ap = float(np.mean(trAp))

    round_va_lg = float(np.mean(vaLg))
    round_va_ag = float(np.mean(vaAg))
    round_va_lp = float(np.mean(vaLp))
    round_va_ap = float(np.mean(vaAp))

    round_kd_g2p = float(np.mean(kd_g2p_vals))

    logs["round"].append(r)
    logs["train_loss_global"].append(round_tr_lg)
    logs["train_acc_global"].append(round_tr_ag)
    logs["train_loss_personal"].append(round_tr_lp)
    logs["train_acc_personal"].append(round_tr_ap)
    logs["val_loss_global"].append(round_va_lg)
    logs["val_acc_global"].append(round_va_ag)
    logs["val_loss_personal"].append(round_va_lp)
    logs["val_acc_personal"].append(round_va_ap)
    logs["kd_g2p"].append(round_kd_g2p)

    print(
        f"[G→P Only KD][Round {r:02d}] "
        f"G(train) loss={round_tr_lg:.3f}, acc={round_tr_ag:.3f} | "
        f"P(train) loss={round_tr_lp:.3f}, acc={round_tr_ap:.3f} || "
        f"G(val) loss={round_va_lg:.3f}, acc={round_va_ag:.3f} | "
        f"P(val) loss={round_va_lp:.3f}, acc={round_va_ap:.3f} | "
        f"KD g->p={round_kd_g2p:.4f}"
    )

    if stopper.step(round_tr_ap):
        print(
            f"Stopping early at round {r:02d} "
            f"(no improvement in personalized training accuracy for {EARLYSTOP_PATIENCE} rounds)."
        )
        break

# ---------------- Final evaluation ----------------
global_test_loss, global_test_acc = eval_head(backbone, server_global_head, test_ds)

spec_losses, spec_accs = [], []
glob_losses, glob_accs = [], []

for client_id in range(NUM_CLIENTS):
    spec_ds_k, _ = make_spec_test_ds_for_client(client_id)

    l_spec, a_spec = eval_head(backbone, personal_heads[client_id], spec_ds_k)
    l_glob, a_glob = eval_head(backbone, personal_heads[client_id], test_ds)

    spec_losses.append(l_spec)
    spec_accs.append(a_spec)
    glob_losses.append(l_glob)
    glob_accs.append(a_glob)

print("\nFinal results")
print(f"Global model on full test set: loss={global_test_loss:.4f}, acc={global_test_acc:.4f}")
print(f"Average personalized model on specialization test: loss={float(np.mean(spec_losses)):.4f}, acc={float(np.mean(spec_accs)):.4f}")
print(f"Average personalized model on full test set: loss={float(np.mean(glob_losses)):.4f}, acc={float(np.mean(glob_accs)):.4f}")

# ---------------- Save logs ----------------
with open(MDL_LOGS_PATH, "w") as f:
    json.dump(logs, f, indent=2)
print("\nSaved logs to:", MDL_LOGS_PATH)

# ---------------- Save metrics CSV ----------------
with open(MDL_METRICS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "round",
        "train_loss_global", "train_acc_global",
        "train_loss_personal", "train_acc_personal",
        "val_loss_global", "val_acc_global",
        "val_loss_personal", "val_acc_personal",
        "kd_g2p",
    ])
    for i in range(len(logs["round"])):
        writer.writerow([
            logs["round"][i],
            logs["train_loss_global"][i], logs["train_acc_global"][i],
            logs["train_loss_personal"][i], logs["train_acc_personal"][i],
            logs["val_loss_global"][i], logs["val_acc_global"][i],
            logs["val_loss_personal"][i], logs["val_acc_personal"][i],
            logs["kd_g2p"][i],
        ])
print("Saved metrics CSV to:", MDL_METRICS_CSV)

# ---------------- Save the final models ----------------
inp = keras.Input(shape=fedavg_model.input_shape[1:])
out = server_global_head(backbone(inp))
global_model = keras.Model(inp, out, name="global_HD_FL_g2p_only")

GLOBAL_MODEL_PATH = os.path.join(SAVE_DIR, "global_HD-FL_g2p_only.h5")
global_model.save(GLOBAL_MODEL_PATH)
print("\nSaved global model to:", GLOBAL_MODEL_PATH)

inputs = keras.Input(shape=(feat_dim,))
outputs = [head(inputs) for head in personal_heads]
personal_model = keras.Model(inputs, outputs, name="personal_HD_FL_g2p_only")

PERSONAL_MODEL_PATH = os.path.join(SAVE_DIR, "person_HD-FL_g2p_only.h5")
personal_model.save(PERSONAL_MODEL_PATH)
print("Saved personalized heads model to:", PERSONAL_MODEL_PATH)

Ablation Study 3 — P→G Only KD

In [ ]:
# =========================================================
#
# Purpose:
# - Initialize from the FedAvg checkpoint saved in Cell 2
# - Rebuild the shared backbone and shared global head
# - Maintain one private personal head per client
# - Train the backbone, global head, and personal head jointly on each client
# - Use only the personal-to-global KD direction
# - Aggregate only the shared backbone and global head across clients
# - Apply early stopping based on average personalized training accuracy
# - Track round-level global/personal training and validation metrics, along with P→G KD
# - Evaluate:
#     (1) the final global model on the full test set
#     (2) the final personalized models on their specialization test sets
#     (3) the final personalized models on the full test set
# - Save the final models, logs, and CSV metrics
#
# Saves:
#   MDL_global_p2g_only_02.h5
#   MDL_heads_p2g_only_h5/
#   MDL_logs_p2g_only_02.json
#   MDL_metrics_p2g_only_rows_02.csv
# ============================================================

import os
import json
import csv
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Required objects from Cell 1 ----------------
required_vars = [
    "client_train_datasets",
    "client_val_datasets",
    "test_ds",
    "client_present",
    "NUM_CLIENTS",
    "NUM_CLASSES",
    "BATCH_SIZE",
    "SEED",
]
for name in required_vars:
    assert name in globals(), f"Missing {name}. Run Cell 1 first."

SAVE_DIR = "/content/drive/MyDrive/...."
MDL_GLOBAL_PATH = f"{SAVE_DIR}/MDL_global_p2g_only_02.h5"
MDL_HEADS_DIR = f"{SAVE_DIR}/MDL_heads_p2g_only_h5"
MDL_LOGS_PATH = f"{SAVE_DIR}/MDL_logs_p2g_only_02.json"
MDL_METRICS_CSV = f"{SAVE_DIR}/MDL_metrics_p2g_only_rows_02.csv"
os.makedirs(MDL_HEADS_DIR, exist_ok=True)

# ---------------- Locate the FedAvg checkpoint ----------------
fedavg_candidates = (
    glob.glob(f"{SAVE_DIR}/MNIST_FedAvg_02.h5") +
    glob.glob(f"{SAVE_DIR}/*/MNIST_FedAvg_02*.h5") +
    glob.glob(f"{SAVE_DIR}/**/MNIST_FedAvg_02*.h5", recursive=True)
)
fedavg_candidates = sorted(set(fedavg_candidates))

if len(fedavg_candidates) == 0:
    raise FileNotFoundError(f"No FedAvg checkpoint found in {SAVE_DIR}")

FEDAVG_PATH = next(
    (p for p in fedavg_candidates if os.path.basename(p) == "MNIST_FedAvg_02.h5"),
    fedavg_candidates[0]
)
print("Using FedAvg checkpoint:", FEDAVG_PATH)

# ---------------- Training configuration ----------------
ROUNDS = 20
BATCHES_PER_ROUND = None
LR_JOINT = 1e-3

TEMP = 2.0
L_P2G = 1.0

EARLYSTOP_PATIENCE = 3
EARLYSTOP_MIN_DELTA = 0.0

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


def load_normalized_mnist_test_like_cell1():
    """
    Rebuild the MNIST test set using the same preprocessing
    and normalization used in Cell 1.
    """
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()

    y_te = y_te.reshape(-1).astype("int32")
    x_tr = (x_tr.astype("float32") / 255.0)[..., None]
    x_te = (x_te.astype("float32") / 255.0)[..., None]

    mean = np.mean(x_tr, axis=(0, 1, 2), keepdims=True)
    std = np.std(x_tr, axis=(0, 1, 2), keepdims=True) + 1e-7

    x_te = (x_te - mean) / std
    return x_te, y_te


x_test_arr, y_test_arr = load_normalized_mnist_test_like_cell1()


def make_spec_test_ds_for_client(client_id):
    """
    Build a client-specific test set containing only the labels
    available to that client.
    """
    allowed = np.array(sorted(set(client_present[client_id])), dtype=np.int32)
    mask = np.isin(y_test_arr, allowed)

    x = x_test_arr[mask]
    y = y_test_arr[mask]

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, int(mask.sum())


def clone_model_with_weights(model):
    """
    Create a copy of a model together with its current weights.
    """
    cloned = keras.models.clone_model(model)
    cloned.build(model.input_shape)
    cloned.set_weights(model.get_weights())
    return cloned


def average_weights(list_of_weight_lists):
    """
    Compute the element-wise mean across a list of model weights.
    """
    return [np.mean(np.stack(w), axis=0) for w in zip(*list_of_weight_lists)]


def kd_kl(p_logits, q_logits, T=2.0):
    """
    Compute KL(softmax(p/T) || softmax(q/T)) scaled by T^2.
    """
    p = tf.nn.softmax(p_logits / T, axis=-1)
    q = tf.nn.softmax(q_logits / T, axis=-1)

    kl = tf.reduce_mean(
        tf.reduce_sum(
            p * (tf.math.log(p + 1e-8) - tf.math.log(q + 1e-8)),
            axis=-1
        )
    )
    return kl * (T * T)


def apply_gradients_safe(optimizer, grads, variables):
    """
    Apply gradients after removing any None entries.
    """
    grads_and_vars = [(g, v) for g, v in zip(grads, variables) if g is not None]
    if grads_and_vars:
        optimizer.apply_gradients(grads_and_vars)


def eval_full_model(model, ds):
    """
    Evaluate a full end-to-end model on a dataset.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        logits = model(x, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def eval_head(backbone, head, ds):
    """
    Evaluate a backbone-head pair on a dataset.
    """
    acc = tf.keras.metrics.SparseCategoricalAccuracy()
    total_loss, total_n = 0.0, 0

    for x, y in ds:
        features = backbone(x, training=False)
        logits = head(features, training=False)
        loss = loss_fn(y, logits)

        bs = int(tf.shape(x)[0].numpy())
        total_loss += float(loss.numpy()) * bs
        total_n += bs
        acc.update_state(y, tf.nn.softmax(logits))

    return float(total_loss / max(total_n, 1)), float(acc.result().numpy())


def local_train_mdl_p2g_only(local_backbone, local_global_head, personal_head, ds,
                             lr=1e-3, batches_limit=None):
    """
    Joint local update with only the personal-to-global KD direction.

    Updated components:
      - backbone
      - local global head
      - personal head

    Objective:
      CE(global) + CE(personal) + KD(personal -> global)
    """
    optimizer = tf.keras.optimizers.Adam(lr)

    acc_g = tf.keras.metrics.SparseCategoricalAccuracy()
    acc_p = tf.keras.metrics.SparseCategoricalAccuracy()

    total_ce_g = 0.0
    total_ce_p = 0.0
    total_kd_p2g = 0.0
    total_n = 0
    step_count = 0

    for x, y in ds:
        with tf.GradientTape() as tape:
            features = local_backbone(x, training=True)
            logits_global = local_global_head(features, training=True)
            logits_personal = personal_head(features, training=True)

            ce_global = loss_fn(y, logits_global)
            ce_personal = loss_fn(y, logits_personal)

            kd_p2g = kd_kl(logits_personal, logits_global, T=TEMP)

            loss = ce_global + ce_personal + (L_P2G * kd_p2g)

        variables = (
            local_backbone.trainable_variables
            + local_global_head.trainable_variables
            + personal_head.trainable_variables
        )
        grads = tape.gradient(loss, variables)
        apply_gradients_safe(optimizer, grads, variables)

        bs = int(tf.shape(x)[0].numpy())
        total_ce_g += float(ce_global.numpy()) * bs
        total_ce_p += float(ce_personal.numpy()) * bs
        total_kd_p2g += float(kd_p2g.numpy()) * bs
        total_n += bs

        acc_g.update_state(y, tf.nn.softmax(logits_global))
        acc_p.update_state(y, tf.nn.softmax(logits_personal))

        step_count += 1
        if batches_limit is not None and step_count >= batches_limit:
            break

    return (
        float(total_ce_g / max(total_n, 1)),
        float(acc_g.result().numpy()),
        float(total_ce_p / max(total_n, 1)),
        float(acc_p.result().numpy()),
        float(total_kd_p2g / max(total_n, 1)),
    )


class TrainAccEarlyStopping:
    """
    Stop training when the monitored personalized training accuracy
    does not improve for a fixed number of rounds.
    """
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.best = None
        self.bad_rounds = 0

    def step(self, value):
        value = float(value)

        if self.best is None:
            self.best = value
            self.bad_rounds = 0
            return False

        improved = (value - self.best) > self.min_delta
        if improved:
            self.best = value
            self.bad_rounds = 0
        else:
            self.bad_rounds += 1

        return self.bad_rounds >= self.patience


stopper = TrainAccEarlyStopping(
    patience=EARLYSTOP_PATIENCE,
    min_delta=EARLYSTOP_MIN_DELTA
)

# ---------------- Load the FedAvg model ----------------
fedavg_model = keras.models.load_model(FEDAVG_PATH, compile=False)
print("Loaded FedAvg model.")

# Reconstruct the shared backbone and global head
backbone = keras.Model(
    fedavg_model.input,
    fedavg_model.get_layer("feat").output,
    name="backbone"
)
feat_dim = backbone.output_shape[-1]

server_global_head = layers.Dense(NUM_CLASSES, name="server_global_head")
server_global_head(tf.zeros([1, feat_dim]))
server_global_head.set_weights(fedavg_model.get_layer("logits").get_weights())

# Confirm that the reconstructed backbone/head path matches the full model
init_loss_full, init_acc_full = eval_full_model(fedavg_model, test_ds)
init_loss_split, init_acc_split = eval_head(backbone, server_global_head, test_ds)

print("\nInitial consistency check")
print(f"Full FedAvg model:       loss={init_loss_full:.4f}, acc={init_acc_full:.4f}")
print(f"Backbone + global head:  loss={init_loss_split:.4f}, acc={init_acc_split:.4f}")

if abs(init_loss_full - init_loss_split) > 1e-5 or abs(init_acc_full - init_acc_split) > 1e-5:
    raise ValueError(
        "Mismatch between the loaded FedAvg model and the reconstructed backbone/head path."
    )

# Initialize one private head per client
personal_heads = []
for client_id in range(NUM_CLIENTS):
    head = layers.Dense(NUM_CLASSES, name=f"personal_head_{client_id}")
    head(tf.zeros([1, feat_dim]))
    head.set_weights(server_global_head.get_weights())
    personal_heads.append(head)

# Metric storage
logs = {
    "round": [],
    "train_loss_global": [],
    "train_acc_global": [],
    "train_loss_personal": [],
    "train_acc_personal": [],
    "val_loss_global": [],
    "val_acc_global": [],
    "val_loss_personal": [],
    "val_acc_personal": [],
    "kd_p2g": [],
}

# ---------------- Federated training loop ----------------
for r in range(ROUNDS):
    client_backbone_ws = []
    client_global_ws = []

    trLg, trAg = [], []
    trLp, trAp = [], []
    vaLg, vaAg = [], []
    vaLp, vaAp = [], []
    kd_p2g_vals = []

    for client_id in range(NUM_CLIENTS):
        local_backbone = clone_model_with_weights(backbone)

        local_global_head = layers.Dense(NUM_CLASSES, name=f"local_global_head_{client_id}")
        local_global_head(tf.zeros([1, feat_dim]))
        local_global_head.set_weights(server_global_head.get_weights())

        l_g, a_g, l_p, a_p, kd_p2g = local_train_mdl_p2g_only(
            local_backbone=local_backbone,
            local_global_head=local_global_head,
            personal_head=personal_heads[client_id],
            ds=client_train_datasets[client_id],
            lr=LR_JOINT,
            batches_limit=BATCHES_PER_ROUND
        )

        lvg, avg = eval_head(local_backbone, local_global_head, client_val_datasets[client_id])
        lvp, avp = eval_head(local_backbone, personal_heads[client_id], client_val_datasets[client_id])

        trLg.append(l_g)
        trAg.append(a_g)
        trLp.append(l_p)
        trAp.append(a_p)

        vaLg.append(lvg)
        vaAg.append(avg)
        vaLp.append(lvp)
        vaAp.append(avp)

        kd_p2g_vals.append(kd_p2g)

        client_backbone_ws.append(local_backbone.get_weights())
        client_global_ws.append(local_global_head.get_weights())

    # Aggregate only the shared components
    backbone.set_weights(average_weights(client_backbone_ws))
    server_global_head.set_weights(average_weights(client_global_ws))

    round_tr_lg = float(np.mean(trLg))
    round_tr_ag = float(np.mean(trAg))
    round_tr_lp = float(np.mean(trLp))
    round_tr_ap = float(np.mean(trAp))

    round_va_lg = float(np.mean(vaLg))
    round_va_ag = float(np.mean(vaAg))
    round_va_lp = float(np.mean(vaLp))
    round_va_ap = float(np.mean(vaAp))

    round_kd_p2g = float(np.mean(kd_p2g_vals))

    logs["round"].append(r)
    logs["train_loss_global"].append(round_tr_lg)
    logs["train_acc_global"].append(round_tr_ag)
    logs["train_loss_personal"].append(round_tr_lp)
    logs["train_acc_personal"].append(round_tr_ap)
    logs["val_loss_global"].append(round_va_lg)
    logs["val_acc_global"].append(round_va_ag)
    logs["val_loss_personal"].append(round_va_lp)
    logs["val_acc_personal"].append(round_va_ap)
    logs["kd_p2g"].append(round_kd_p2g)

    print(
        f"[P→G Only KD][Round {r:02d}] "
        f"G(train) loss={round_tr_lg:.3f}, acc={round_tr_ag:.3f} | "
        f"P(train) loss={round_tr_lp:.3f}, acc={round_tr_ap:.3f} || "
        f"G(val) loss={round_va_lg:.3f}, acc={round_va_ag:.3f} | "
        f"P(val) loss={round_va_lp:.3f}, acc={round_va_ap:.3f} | "
        f"KD p->g={round_kd_p2g:.4f}"
    )

    if stopper.step(round_tr_ap):
        print(
            f"Stopping early at round {r:02d} "
            f"(no improvement in personalized training accuracy for {EARLYSTOP_PATIENCE} rounds)."
        )
        break

# ---------------- Final evaluation ----------------
global_test_loss, global_test_acc = eval_head(backbone, server_global_head, test_ds)

spec_losses, spec_accs = [], []
glob_losses, glob_accs = [], []

for client_id in range(NUM_CLIENTS):
    spec_ds_k, _ = make_spec_test_ds_for_client(client_id)

    l_spec, a_spec = eval_head(backbone, personal_heads[client_id], spec_ds_k)
    l_glob, a_glob = eval_head(backbone, personal_heads[client_id], test_ds)

    spec_losses.append(l_spec)
    spec_accs.append(a_spec)
    glob_losses.append(l_glob)
    glob_accs.append(a_glob)

print("\nFinal results")
print(f"Global model on full test set: loss={global_test_loss:.4f}, acc={global_test_acc:.4f}")
print(f"Average personalized model on specialization test: loss={float(np.mean(spec_losses)):.4f}, acc={float(np.mean(spec_accs)):.4f}")
print(f"Average personalized model on full test set: loss={float(np.mean(glob_losses)):.4f}, acc={float(np.mean(glob_accs)):.4f}")

# ---------------- Save logs ----------------
with open(MDL_LOGS_PATH, "w") as f:
    json.dump(logs, f, indent=2)
print("\nSaved logs to:", MDL_LOGS_PATH)

# ---------------- Save metrics CSV ----------------
with open(MDL_METRICS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "round",
        "train_loss_global", "train_acc_global",
        "train_loss_personal", "train_acc_personal",
        "val_loss_global", "val_acc_global",
        "val_loss_personal", "val_acc_personal",
        "kd_p2g",
    ])
    for i in range(len(logs["round"])):
        writer.writerow([
            logs["round"][i],
            logs["train_loss_global"][i], logs["train_acc_global"][i],
            logs["train_loss_personal"][i], logs["train_acc_personal"][i],
            logs["val_loss_global"][i], logs["val_acc_global"][i],
            logs["val_loss_personal"][i], logs["val_acc_personal"][i],
            logs["kd_p2g"][i],
        ])
print("Saved metrics CSV to:", MDL_METRICS_CSV)

# ---------------- Save the final models ----------------
inp = keras.Input(shape=fedavg_model.input_shape[1:])
out = server_global_head(backbone(inp))
global_model = keras.Model(inp, out, name="global_HD_FL_p2g_only")

GLOBAL_MODEL_PATH = os.path.join(SAVE_DIR, "global_HD-FL_p2g_only.h5")
global_model.save(GLOBAL_MODEL_PATH)
print("\nSaved global model to:", GLOBAL_MODEL_PATH)

inputs = keras.Input(shape=(feat_dim,))
outputs = [head(inputs) for head in personal_heads]
personal_model = keras.Model(inputs, outputs, name="personal_HD_FL_p2g_only")

PERSONAL_MODEL_PATH = os.path.join(SAVE_DIR, "person_HD-FL_p2g_only.h5")
personal_model.save(PERSONAL_MODEL_PATH)
print("Saved personalized heads model to:", PERSONAL_MODEL_PATH)